<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase44_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 44 — standards snapshot and Arm A schema-binding preflight



This phase stops adding self-authored synthetic detector cases. It automatically finds the verified Phase 43 artifacts in Google Drive, then creates a reproducible public standards snapshot for:

- **OMOP CDM v5.5** — field/table definitions from `OHDSI/CommonDataModel`
- **OHDSI Medical Imaging CDM (MI-CDM)** — the current public ImageWG schema artifacts, including the 1.0 SQL Server DDL and imaging field-level CSV

On the first successful run, the exact Git commit SHAs are written to a persistent **standards lock** in Drive. Later reruns reuse those commits rather than silently moving with `main`.

### What Phase 44 can establish
It can show, from the pinned public schema itself, which target fields exist and which source/evidence gaps still prevent an honest official Arm A load.

### What Phase 44 cannot establish
It does **not** invent Athena concept IDs, approve human adjudication, manufacture real clinical procedures or DICOM UIDs, process ADNI/real MRI, execute official Arm A, execute frozen generic TFL Arm C, or provide independent validation.


In [1]:
# CODE CELL 1/7 — Mount Drive, constants, helpers
import os, re, csv, json, hashlib, shutil, tempfile, zipfile, time, io
import datetime as dt
from pathlib import Path

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='44_public_standards_snapshot_and_armA_schema_binding_preflight'

EXPECTED_P43_SHAREABLE='c24b174761f1e18e92c4d5abf761f7ff5d7c5f09740c8a7e3d0f63a9af50495f'
EXPECTED_P43_METHOD_LOCK='73a709b83dde4490aeac52ab35b150776e93499ac110a57e255e8cca2c459abc'
EXPECTED_GATES='4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'
EXPECTED_CONTRACT='d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'

OMOP_VERSION='5.5'
MICDM_DECLARED_DDL_VERSION='1.0'

OMOP_REPO='OHDSI/CommonDataModel'
IMAGEWG_REPO='OHDSI/ImageWG'

OMOP_FIELD_PATH='inst/csv/OMOP_CDMv5.5_Field_Level.csv'
OMOP_TABLE_PATH='inst/csv/OMOP_CDMv5.5_Table_Level.csv'
MICDM_FIELD_PATH='inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
MICDM_DDL_PATH='inst/ddl/1.0/Sql Server/Medical_Imaging_Ext_sql_server_1.0_ddl.rtf'

def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

if os.environ.get('PHASE44_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE44_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir(), 'Drive mounted, but MyDrive was not readable.'
    ROOTS=[MYDRIVE]
    shared=Path('/content/drive/Shareddrives')
    if shared.is_dir():
        ROOTS.append(shared)

hint=os.environ.get('PHASE44_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute():
        hp=MYDRIVE/hp
    if not hp.is_dir():
        raise FileNotFoundError('Drive hint does not exist: '+hint)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase44_'))
INPUT=WORK/'inputs'; INPUT.mkdir()
STAGE=WORK/'outputs'; STAGE.mkdir()
STD=WORK/'standards'; STD.mkdir()

BASE=Path(os.environ.get('PHASE44_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase44')))
LOCK_PATH=BASE/'ICHI2027_Phase44_STANDARDS_LOCK.json'
RUN_DIR=BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

print('Search roots:',*[str(x) for x in ROOTS],sep='\n  ')
print('Run output:',RUN_DIR)


Mounted at /content/drive
Search roots:
  /content/drive/MyDrive
Run output: /content/drive/MyDrive/ICHI2027/Phase44/run_20261001T013055_051068Z


In [2]:
# CODE CELL 2/7 — Auto-find and cryptographically verify Phase 43 inputs
REQUIRED={
 'summary':('ICHI2027_Phase43_SHAREABLE','.json',EXPECTED_P43_SHAREABLE),
 'method_lock':('ICHI2027_Phase43_METHOD_LOCK','.json',EXPECTED_P43_METHOD_LOCK),
 'gates':('ICHI2027_Phase43_ARM_A_GATES_UNCHANGED','.csv',EXPECTED_GATES),
 'contract':('ICHI2027_Phase43_LOGICAL_CONTRACT_UNCHANGED','.csv',EXPECTED_CONTRACT),
}

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

found={}
for key,(stem,ext,digest) in REQUIRED.items():
    candidates=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for f in files:
                if name_matches(f,stem,ext):
                    p=Path(here)/f
                    try:
                        if sha256_file(p)==digest:
                            candidates.append(p)
                    except OSError:
                        pass
    if not candidates:
        raise FileNotFoundError(f'No SHA-approved Drive copy found for {stem}{ext}; no manual-upload fallback is used.')
    found[key]=sorted(candidates,key=str)[0]

summary43=json.loads(found['summary'].read_text('utf-8'))
method43=json.loads(found['method_lock'].read_text('utf-8'))
with found['gates'].open(newline='',encoding='utf-8') as f:
    gates=list(csv.DictReader(f))
with found['contract'].open(newline='',encoding='utf-8') as f:
    contract=list(csv.DictReader(f))

assert summary43['phase']=='43_frozen_method_synthetic_holdout_no_retuning_audit'
assert summary43['holdout_matched_authored_fault_events']==240
assert summary43['holdout_unexpected_alert_events']==0
assert summary43['method_retuned_after_holdout_generation'] is False
assert summary43['official_omop_etl_executed'] is False
assert summary43['official_micdm_load_executed'] is False
assert summary43['armA_unapproved_original_gates']==10
assert len(gates)==10
assert len(contract)==8

print('PASS: Phase 43 chain verified.')
print('Frozen detector method:',method43['method_version'])
print('Open Arm A gates:',len(gates))


PASS: Phase 43 chain verified.
Frozen detector method: phase42_internal_source_projection_v0.1
Open Arm A gates: 10


## Pin public standards to immutable Git commits

The first successful run resolves `main` once and then downloads files through the exact commit SHA. The lock is persisted outside the timestamped run directory so a rerun does not silently change the research target.

If a standards lock already exists, its commits are reused.


In [3]:
# CODE CELL 3/7 — Resolve/reuse immutable standards commits and download exact files
import requests

session=requests.Session()
session.headers.update({'User-Agent':'ICHI2027-neuroimaging-fhir-omop-fidelity-phase44'})

def github_commit(repo,ref='main'):
    url=f'https://api.github.com/repos/{repo}/commits/{ref}'
    r=session.get(url,timeout=60)
    r.raise_for_status()
    j=r.json()
    sha=j['sha']
    assert re.fullmatch(r'[0-9a-f]{40}',sha)
    return sha,j.get('commit',{}).get('committer',{}).get('date')

BASE.mkdir(parents=True,exist_ok=True)

if LOCK_PATH.exists():
    standards_lock=json.loads(LOCK_PATH.read_text('utf-8'))
    assert standards_lock['omop_version']==OMOP_VERSION
    assert standards_lock['micdm_declared_ddl_version']==MICDM_DECLARED_DDL_VERSION
    omop_commit=standards_lock['omop_repo_commit']
    imagewg_commit=standards_lock['imagewg_repo_commit']
    lock_origin='REUSED_EXISTING_DRIVE_LOCK'
else:
    omop_commit,omop_commit_date=github_commit(OMOP_REPO)
    imagewg_commit,imagewg_commit_date=github_commit(IMAGEWG_REPO)
    standards_lock={
      'project':PROJECT,
      'phase_created':PHASE,
      'created_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
      'omop_version':OMOP_VERSION,
      'micdm_declared_ddl_version':MICDM_DECLARED_DDL_VERSION,
      'omop_repo':OMOP_REPO,
      'omop_repo_commit':omop_commit,
      'omop_repo_commit_date':omop_commit_date,
      'imagewg_repo':IMAGEWG_REPO,
      'imagewg_repo_commit':imagewg_commit,
      'imagewg_repo_commit_date':imagewg_commit_date,
      'warning':'Commit pinning fixes public schema files; it does not approve vocabulary concepts or source evidence.'
    }
    tmp_lock=WORK/'standards_lock_pending.json'
    tmp_lock.write_text(json.dumps(standards_lock,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    lock_origin='NEW_LOCK_PENDING_VALIDATION'

def raw_url(repo,commit,path):
    from urllib.parse import quote
    # quote path pieces but retain slash separators
    safe='/'.join(quote(x,safe='') for x in path.split('/'))
    return f'https://raw.githubusercontent.com/{repo}/{commit}/{safe}'

downloads=[
 ('OMOP_CDMv5.5_Field_Level.csv',OMOP_REPO,omop_commit,OMOP_FIELD_PATH),
 ('OMOP_CDMv5.5_Table_Level.csv',OMOP_REPO,omop_commit,OMOP_TABLE_PATH),
 ('OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv',IMAGEWG_REPO,imagewg_commit,MICDM_FIELD_PATH),
 ('Medical_Imaging_Ext_sql_server_1.0_ddl.rtf',IMAGEWG_REPO,imagewg_commit,MICDM_DDL_PATH),
]

download_manifest=[]
offline_fixture=os.environ.get('PHASE44_OFFLINE_STANDARDS_DIR','').strip()
for out_name,repo,commit,path in downloads:
    target=STD/out_name
    if offline_fixture:
        src=Path(offline_fixture)/out_name
        if not src.is_file():
            raise FileNotFoundError('Offline fixture missing: '+str(src))
        shutil.copyfile(src,target)
        source_url='OFFLINE_TEST_FIXTURE'
    else:
        source_url=raw_url(repo,commit,path)
        r=session.get(source_url,timeout=90)
        r.raise_for_status()
        target.write_bytes(r.content)
    if target.stat().st_size==0:
        raise RuntimeError('Downloaded empty standards file: '+out_name)
    download_manifest.append({
      'artifact':out_name,'repository':repo,'commit':commit,'repository_path':path,
      'sha256':sha256_file(target),'bytes':target.stat().st_size,'source_url':source_url
    })

print('OMOP commit:',omop_commit)
print('ImageWG commit:',imagewg_commit)
print('Lock status:',lock_origin)
for x in download_manifest:
    print(x['artifact'],x['sha256'],x['bytes'])


OMOP commit: 4a910305b2cb74a4fc2b2c34baf44eb0542ff03f
ImageWG commit: 8f0a19bb94e941bb335a097bc9851fab672621c7
Lock status: NEW_LOCK_PENDING_VALIDATION
OMOP_CDMv5.5_Field_Level.csv 7089ca523c13a29ec42ae55d4ef857a67bde0718a777c221858fad029b939994 145278
OMOP_CDMv5.5_Table_Level.csv 42553b33bd7ec7085b0bc8df029fc6fd83aaa12650686bb259c4d492a10eb95d 48930
OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4 4982
Medical_Imaging_Ext_sql_server_1.0_ddl.rtf 18f705becae5ddaab2c3d2c1bc56d6960f62446cf29c1cbf7e2a457ff74315f8 1468


In [4]:
# CODE CELL 4/7 — Parse pinned OMOP/MI-CDM schemas and verify required structures

def normalized_dict_rows(path):
    text=Path(path).read_text('utf-8-sig',errors='replace')
    rdr=csv.DictReader(io.StringIO(text))
    rows=[]
    for row in rdr:
        clean={}
        for k,v in row.items():
            nk=(k or '').replace('\ufeff','').replace('\xa0',' ').strip()
            nv=(v or '').replace('\xa0',' ').strip()
            clean[nk]=nv
        rows.append(clean)
    return rows

omop_fields=normalized_dict_rows(STD/'OMOP_CDMv5.5_Field_Level.csv')
omop_tables=normalized_dict_rows(STD/'OMOP_CDMv5.5_Table_Level.csv')
micdm_fields=normalized_dict_rows(STD/'OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv')

def field(row):
    return (row.get('cdmFieldName') or '').strip().lower()
def table(row):
    return (row.get('cdmTableName') or '').strip().upper()
def required(row):
    for k,v in row.items():
        if k.strip().lower().startswith('isrequired'):
            return v.strip().lower()=='yes'
    return False

omop_table_names={table(r) for r in omop_fields}
micdm_table_names={table(r) for r in micdm_fields}

assert 'PERSON' in omop_table_names
assert 'MEASUREMENT' in omop_table_names
assert 'PROCEDURE_OCCURRENCE' in omop_table_names
assert {'IMAGE_OCCURRENCE','IMAGE_FEATURE'} <= micdm_table_names

io_fields={field(r):r for r in micdm_fields if table(r)=='IMAGE_OCCURRENCE'}
if_fields={field(r):r for r in micdm_fields if table(r)=='IMAGE_FEATURE'}

required_io={'image_occurrence_id','person_id','procedure_occurrence_id','image_occurrence_date',
             'image_study_uid','image_series_uid','modality_concept_id'}
required_if={'image_feature_id','person_id','image_occurrence_id',
             'image_feature_concept_id','image_feature_type_concept_id'}

missing_io=required_io-set(io_fields)
missing_if=required_if-set(if_fields)
assert not missing_io,missing_io
assert not missing_if,missing_if

required_io_actual=sorted(k for k,r in io_fields.items() if required(r))
required_if_actual=sorted(k for k,r in if_fields.items() if required(r))

ddl_bytes=(STD/'Medical_Imaging_Ext_sql_server_1.0_ddl.rtf').read_bytes()
ddl_text=ddl_bytes.decode('latin-1',errors='ignore').upper()
assert 'IMAGE_OCCURRENCE' in ddl_text
assert 'IMAGE_FEATURE' in ddl_text

print('PASS: pinned public schemas parsed.')
print('MI-CDM IMAGE_OCCURRENCE fields:',len(io_fields),'required:',required_io_actual)
print('MI-CDM IMAGE_FEATURE fields:',len(if_fields),'required:',required_if_actual)
print('OMOP PERSON/MEASUREMENT/PROCEDURE_OCCURRENCE confirmed in v5.5 field definitions.')


PASS: pinned public schemas parsed.
MI-CDM IMAGE_OCCURRENCE fields: 11 required: ['image_occurrence_date', 'image_occurrence_id', 'image_series_uid', 'image_study_uid', 'modality_concept_id', 'person_id', 'procedure_occurrence_id']
MI-CDM IMAGE_FEATURE fields: 12 required: ['image_feature_concept_id', 'image_feature_id', 'image_feature_type_concept_id', 'image_occurrence_id', 'person_id']
OMOP PERSON/MEASUREMENT/PROCEDURE_OCCURRENCE confirmed in v5.5 field definitions.


## Schema-based preflight

This step compares the existing eight-dimension logical contract with the pinned schema. It deliberately distinguishes:

- **schema support exists**
- **schema exists but source evidence is absent**
- **schema exists but vocabulary concepts are not yet approved**
- **information still requires the versioned fidelity sidecar**

A public table definition cannot itself clear private Athena/human/source-evidence gates.


In [5]:
# CODE CELL 5/7 — Build evidence-based Arm A schema-binding preflight

def yesno(x): return 'YES' if x else 'NO'

preflight=[
 {
  'dimension':'patient_link',
  'pinned_target':'OMOP PERSON',
  'schema_evidence':'PERSON exists in OMOP CDM v5.5',
  'source_evidence':'19 deterministic synthetic people; source gender/birthdate absent',
  'preflight_status':'SCHEMA_SUPPORTED_POLICY_GATE_OPEN',
  'blocking_reason':'Independent policy for absent gender/birthdate remains unapproved'
 },
 {
  'dimension':'image_study',
  'pinned_target':'MI-CDM IMAGE_OCCURRENCE',
  'schema_evidence':'Required person_id, procedure_occurrence_id, image_occurrence_date, image_study_UID, image_series_UID, modality_concept_id are present',
  'source_evidence':'38 synthetic studies; no source-attested clinical Procedure and no attested real DICOM UID',
  'preflight_status':'SCHEMA_PINNED_SOURCE_EVIDENCE_BLOCKED',
  'blocking_reason':'Pinned schema requires procedure_occurrence_id and DICOM study/series UID fields; current clinical Procedure/real UID evidence is absent'
 },
 {
  'dimension':'image_measure',
  'pinned_target':'MI-CDM IMAGE_FEATURE + OMOP MEASUREMENT when approved',
  'schema_evidence':'IMAGE_FEATURE requires image_feature_concept_id and image_feature_type_concept_id; MEASUREMENT exists',
  'source_evidence':'148 synthetic volume observations with source anatomy/laterality/value/unit',
  'preflight_status':'SCHEMA_PINNED_VOCABULARY_BLOCKED',
  'blocking_reason':'Required standard/type/anatomy/modality bindings are not independently approved; do not invent concept IDs'
 },
 {
  'dimension':'extraction_model',
  'pinned_target':'MI-CDM IMAGE_FEATURE.alg_system + fidelity sidecar as needed',
  'schema_evidence':'alg_system exists and is described as algorithm URI including version information',
  'source_evidence':'Synthetic source retains algorithm name/version and provenance',
  'preflight_status':'SCHEMA_FIELD_AVAILABLE_BINDING_REVIEW_REQUIRED',
  'blocking_reason':'Exact representation/URI convention has not been independently adjudicated'
 },
 {
  'dimension':'reliability',
  'pinned_target':'Versioned fidelity sidecar; target linkage where supported',
  'schema_evidence':'No dedicated TRACE-state field is established by the inspected IMAGE_FEATURE field list',
  'source_evidence':'Original synthetic TRACE states retained; Phase 42 method frozen and Phase 43 no-retuning audit completed',
  'preflight_status':'SIDECAR_REQUIRED',
  'blocking_reason':'TRACE state/reason/version must remain machine-readable without pretending MI-CDM natively defines TRACE'
 },
 {
  'dimension':'longitudinal',
  'pinned_target':'IMAGE_OCCURRENCE ↔ IMAGE_FEATURE plus source timepoint/date mapping',
  'schema_evidence':'IMAGE_FEATURE.image_occurrence_id links to IMAGE_OCCURRENCE; imaging date is present',
  'source_evidence':'Synthetic baseline/12-month links exist',
  'preflight_status':'SCHEMA_LINK_AVAILABLE_SOURCE_BINDING_REVIEW_REQUIRED',
  'blocking_reason':'Official target pairing must be tested only after approved source/target binding'
 },
 {
  'dimension':'procedures',
  'pinned_target':'OMOP PROCEDURE_OCCURRENCE referenced by IMAGE_OCCURRENCE',
  'schema_evidence':'procedure_occurrence_id is required in inspected IMAGE_OCCURRENCE field definition',
  'source_evidence':'Only post-hoc manufactured synthetic Procedure fixtures; source clinical procedures attested = 0',
  'preflight_status':'BLOCKED_SOURCE_ABSENT',
  'blocking_reason':'Cannot promote manufactured fixture to source-attested clinical Procedure'
 },
 {
  'dimension':'dicom_uid',
  'pinned_target':'IMAGE_OCCURRENCE.image_study_UID + image_series_UID',
  'schema_evidence':'Both UID fields are required in inspected IMAGE_OCCURRENCE field definition',
  'source_evidence':'Current DICOM-style study/series identifiers are synthetic; real DICOM UIDs attested = 0',
  'preflight_status':'BLOCKED_SOURCE_ABSENT',
  'blocking_reason':'Cannot claim real imaging-source UID provenance'
 },
]
assert len(preflight)==8

schema_findings=[
 {'finding':'OMOP_VERSION_PINNED','value':OMOP_VERSION,'status':'PASS'},
 {'finding':'OMOP_COMMIT_PINNED','value':omop_commit,'status':'PASS'},
 {'finding':'MICDM_DECLARED_DDL_VERSION','value':MICDM_DECLARED_DDL_VERSION,'status':'PASS'},
 {'finding':'IMAGEWG_COMMIT_PINNED','value':imagewg_commit,'status':'PASS'},
 {'finding':'IMAGE_OCCURRENCE_PRESENT','value':yesno('IMAGE_OCCURRENCE' in micdm_table_names),'status':'PASS'},
 {'finding':'IMAGE_FEATURE_PRESENT','value':yesno('IMAGE_FEATURE' in micdm_table_names),'status':'PASS'},
 {'finding':'IMAGE_OCCURRENCE_PROCEDURE_REQUIRED','value':yesno(required(io_fields['procedure_occurrence_id'])),'status':'PASS'},
 {'finding':'IMAGE_OCCURRENCE_STUDY_UID_REQUIRED','value':yesno(required(io_fields['image_study_uid'])),'status':'PASS'},
 {'finding':'IMAGE_OCCURRENCE_SERIES_UID_REQUIRED','value':yesno(required(io_fields['image_series_uid'])),'status':'PASS'},
 {'finding':'IMAGE_FEATURE_CONCEPT_REQUIRED','value':yesno(required(if_fields['image_feature_concept_id'])),'status':'PASS'},
 {'finding':'IMAGE_FEATURE_TYPE_CONCEPT_REQUIRED','value':yesno(required(if_fields['image_feature_type_concept_id'])),'status':'PASS'},
 {'finding':'IMAGE_FEATURE_ALG_SYSTEM_AVAILABLE','value':yesno('alg_system' in if_fields),'status':'PASS'},
]

# Original ten gates are carried forward unchanged.
assert len(gates)==10 and all(r['official_validation_passed']=='False' for r in gates)

print('PASS: eight-dimension preflight generated.')
for r in preflight:
    print(f"{r['dimension']}: {r['preflight_status']}")
print('Original Arm A gates remain open:',len(gates))


PASS: eight-dimension preflight generated.
patient_link: SCHEMA_SUPPORTED_POLICY_GATE_OPEN
image_study: SCHEMA_PINNED_SOURCE_EVIDENCE_BLOCKED
image_measure: SCHEMA_PINNED_VOCABULARY_BLOCKED
extraction_model: SCHEMA_FIELD_AVAILABLE_BINDING_REVIEW_REQUIRED
reliability: SIDECAR_REQUIRED
longitudinal: SCHEMA_LINK_AVAILABLE_SOURCE_BINDING_REVIEW_REQUIRED
procedures: BLOCKED_SOURCE_ABSENT
dicom_uid: BLOCKED_SOURCE_ABSENT
Original Arm A gates remain open: 10


In [6]:
# CODE CELL 6/7 — Persist the standards lock only after schema validation; write public-safe artifacts

def write_csv(path,rows):
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

# If this is the first run, persist the validated lock now.
if not LOCK_PATH.exists():
    LOCK_PATH.parent.mkdir(parents=True,exist_ok=True)
    tmp=WORK/'lock_write.json'
    tmp.write_text(json.dumps(standards_lock,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    shutil.copyfile(tmp,LOCK_PATH)
    if sha256_file(tmp)!=sha256_file(LOCK_PATH):
        raise RuntimeError('Drive standards-lock readback hash mismatch.')
    lock_origin='NEW_VALIDATED_LOCK_CREATED'
else:
    lock_origin='REUSED_EXISTING_DRIVE_LOCK'

manifest_path=STAGE/'ICHI2027_Phase44_STANDARDS_ARTIFACT_MANIFEST.csv'
write_csv(manifest_path,download_manifest)

preflight_path=STAGE/'ICHI2027_Phase44_SCHEMA_BINDING_PREFLIGHT.csv'
write_csv(preflight_path,preflight)

findings_path=STAGE/'ICHI2027_Phase44_SCHEMA_FINDINGS.csv'
write_csv(findings_path,schema_findings)

gates_path=STAGE/'ICHI2027_Phase44_ARM_A_GATES_UNCHANGED.csv'
write_csv(gates_path,gates)

contract_path=STAGE/'ICHI2027_Phase44_LOGICAL_CONTRACT_CARRIED_FORWARD.csv'
write_csv(contract_path,contract)

lock_copy=STAGE/'ICHI2027_Phase44_STANDARDS_LOCK.json'
lock_copy.write_text(json.dumps(standards_lock,sort_keys=True,indent=2)+'\n',encoding='utf-8')

readme=STAGE/'ICHI2027_Phase44_README.txt'
readme.write_text(
  'PHASE 44 pins public OMOP CDM v5.5 and OHDSI ImageWG/MI-CDM schema artifacts to immutable Git commits.\n'
  'It performs schema-binding preflight only. It does not execute an official OMOP/MI-CDM ETL.\n'
  'It does not approve Athena concepts, human adjudication, source clinical Procedures, or real DICOM UIDs.\n'
  'The inspected MI-CDM IMAGE_OCCURRENCE field definition requires procedure_occurrence_id, image_study_UID, image_series_UID, and modality_concept_id.\n'
  'The inspected IMAGE_FEATURE field definition requires image_feature_concept_id and image_feature_type_concept_id and includes optional alg_system.\n'
  'All ten original Arm A evidence/execution gates remain open.\n',
  encoding='utf-8'
)

artifact_paths=[manifest_path,preflight_path,findings_path,gates_path,contract_path,lock_copy,readme]


In [7]:
# CODE CELL 7/7 — Create shareable summary, save to Drive, and verify every copied hash

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'scope':'PUBLIC_STANDARDS_SCHEMA_PREFLIGHT_PLUS_SYNTHETIC_PRIOR_CHAIN',
 'input_phase43_shareable_sha256':sha256_file(found['summary']),
 'input_phase43_method_lock_sha256':sha256_file(found['method_lock']),
 'standards_lock_origin':lock_origin,
 'omop_version':OMOP_VERSION,
 'omop_repo_commit':omop_commit,
 'micdm_declared_ddl_version':MICDM_DECLARED_DDL_VERSION,
 'imagewg_repo_commit':imagewg_commit,
 'schema_binding_dimensions':8,
 'image_occurrence_present':True,
 'image_feature_present':True,
 'image_occurrence_procedure_occurrence_id_required':required(io_fields['procedure_occurrence_id']),
 'image_occurrence_study_uid_required':required(io_fields['image_study_uid']),
 'image_occurrence_series_uid_required':required(io_fields['image_series_uid']),
 'image_feature_concept_id_required':required(if_fields['image_feature_concept_id']),
 'image_feature_type_concept_id_required':required(if_fields['image_feature_type_concept_id']),
 'image_feature_alg_system_available':('alg_system' in if_fields),
 'source_clinical_procedures_attested':0,
 'real_dicom_uid_source_attested':0,
 'approved_vocabulary_bindings':0,
 'armA_unapproved_original_gates':10,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'frozen_original_generic_tfl_armC_executed':False,
 'independently_blinded_external_validation':False,
 'real_mri_rows_processed':0,
 'full_A_B_C_D_evaluation':False,
 'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED',
 'standards_artifact_sha256':{x['artifact']:x['sha256'] for x in download_manifest},
 'artifact_sha256':{p.name:sha256_file(p) for p in artifact_paths},
 'warning':'Pinned public schema support does not substitute for private vocabulary adjudication, source evidence, official ETL execution, or external validation.'
}

summary_path=STAGE/'ICHI2027_Phase44_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifact_paths.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase44_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifact_paths:
        z.write(p,p.name)

RUN_DIR.mkdir(parents=True,exist_ok=False)
for p in artifact_paths+[zip_path]:
    dest=RUN_DIR/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256_file(dest)==sha256_file(p)
            break
        except Exception as e:
            if attempt==2:
                raise RuntimeError('Drive save/readback verification failed: '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 44 standards snapshot and schema-binding preflight complete.')
print('Pinned OMOP v5.5 commit:',omop_commit)
print('Pinned ImageWG commit:',imagewg_commit)
print('Original Arm A gates still open: 10')
print('Output folder:',RUN_DIR)
print('Shareable:',RUN_DIR/summary_path.name)
print('ZIP:',RUN_DIR/zip_path.name)


SUCCESS: Phase 44 standards snapshot and schema-binding preflight complete.
Pinned OMOP v5.5 commit: 4a910305b2cb74a4fc2b2c34baf44eb0542ff03f
Pinned ImageWG commit: 8f0a19bb94e941bb335a097bc9851fab672621c7
Original Arm A gates still open: 10
Output folder: /content/drive/MyDrive/ICHI2027/Phase44/run_20261001T013055_051068Z
Shareable: /content/drive/MyDrive/ICHI2027/Phase44/run_20261001T013055_051068Z/ICHI2027_Phase44_SHAREABLE.json
ZIP: /content/drive/MyDrive/ICHI2027/Phase44/run_20261001T013055_051068Z/ICHI2027_Phase44_PUBLIC_OUTPUTS.zip


## Interpretation after a successful run

Phase 44 should be treated as **standards pinning + schema evidence**, not as Arm A execution.

A particularly important check is whether the pinned MI-CDM schema itself requires source evidence that this project does not currently possess. If `procedure_occurrence_id`, `image_study_UID`, and `image_series_UID` are required while the current source only has manufactured Procedure fixtures and synthetic UID-style identifiers, the correct scientific response is to keep official Arm A blocked rather than populate those fields with invented clinical evidence.

The next phase should be chosen from the remaining real gates—not another synthetic detector phase.
